<a href="https://colab.research.google.com/github/427paul/ai_agent/blob/main/ai_agent_09_LangSmith.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U "langchain==0.3.*" "langchain-core==0.3.*" "langchain-community==0.3.*" "langgraph==0.3.*" "langchain-huggingface" "huggingface_hub" "sentence-transformers" wikipedia -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '/content/drive/MyDrive/LangGraph/'

# API 키 로드 및 환경변수 설정
load_api_keys(path + 'api_key.txt')

In [ ]:
pip install -U langsmith

# Tracing Project

In [ ]:
import os

# 1. LangSmith 설정
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "" # 발급받은 키 입력
os.environ["LANGSMITH_PROJECT"] = "Hello World" # 프로젝트 이름

# 2. HuggingFace 설정
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "" # OpenAI 키 입력

In [ ]:
from langsmith import traceable
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# --- 코드 실행 ---

# OpenAI 클라이언트를 LangSmith 래퍼로 감쌉니다.
# 이렇게 하면 자동으로 API 호출이 트래킹됩니다.
client = wrap_openai(openai.Client())

@traceable(run_type="tool", name="Retrieve Context")
def my_tool(question: str) -> str:
    return "During this morning's meeting, we solved all world conflict."

@traceable(name="Chat Pipeline")
def chat_pipeline(question: str):
    context = my_tool(question)
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Please respond to the user's request only based on the given context.",
        },
        {
            "role": "user",
            "content": f"Question: {question}\nContext: {context}",
        },
    ]
    # wrap_openai 덕분에 이 호출은 자동으로 기록됩니다.
    chat_completion = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages
    )
    return chat_completion.choices[0].message.content

# 실행!
result = chat_pipeline("Can you summarize this morning's meetings?")
print(f"답변: {result}")

In [ ]:
from langsmith import traceable
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# --- 코드 실행 ---

# 2. 모델 설정
llm_ep = HuggingFaceEndpoint(repo_id="Qwen/Qwen2.5-7B-Instruct", task="text-generation")
model = ChatHuggingFace(llm=llm_ep)

# 3. 도구 정의 (Traceable 추가)
@traceable(run_type="tool", name="Knowledge-Base")
def my_tool(question: str) -> str:
    return "During this morning's meeting, we solved all world conflict."

# 4. 전체 파이프라인 정의 (Traceable 추가)
@traceable(name="HF-Chat-Pipeline")
def chat_pipeline(question: str):
    context = my_tool(question)

    # LangChain 모델 호출 (이 과정도 자동으로 추적됩니다)
    messages = [
        ("system", "You are a helpful assistant based on the context."),
        ("user", f"Question: {question}\nContext: {context}")
    ]

    # .invoke()를 사용하면 LangChain이 내부적으로 LangSmith와 연동됩니다.
    response = model.invoke(messages)
    return response.content

# 실행!

result = chat_pipeline("Can you summarize this morning's meetings?")
print(f"Hugging Face 답변: {result}")

# Create Prompt

In [ ]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate

# Connect to the LangSmith client

client = Client()

# Define the prompt

prompt = ChatPromptTemplate([
    ("system", "You are a helpful chatbot. Answer the question as best as you can. provide the answer within 1 line"),
    ("user", "{question}"),
])

# Push the prompt
client.push_prompt("my-fistprompt", object=prompt)

# Pull Prompt

In [ ]:
from langsmith import Client
from openai import OpenAI
from langchain_core.messages import convert_to_openai_messages

# Connect to LangSmith and OpenAI

client = Client()
oai_client = OpenAI()

# Pull the prompt to use

# You can also specify a specific commit by passing the commit hash "my-prompt:<commit-hash>"

prompt = client.pull_prompt("my-fistprompt")

# Since our prompt
# only has one variable we could also pass in the value directly

# The code below is equivalent to formatted_prompt = prompt.invoke("What is the color of the sky?")

formatted_prompt = prompt.invoke({"question": "What is the color of the sky?"})

# Test the prompt

response = oai_client.chat.completions.create(
    model="gpt-4o",
    messages=convert_to_openai_messages(formatted_prompt.messages), # type: ignore
)
print(response)

In [ ]:
import os
from langsmith import Client
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# 2. 모델 설정 (Qwen2.5-7B 또는 Llama-3.2 추천)
llm_ep = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.1
)
model = ChatHuggingFace(llm=llm_ep)

# 3. LangSmith 클라이언트 생성 및 프롬프트 Pull
client = Client()
# Hub에서 프롬프트를 가져옵니다.
prompt = client.pull_prompt("my-fistprompt")

# 4. 프롬프트 포맷팅
# 입력 변수가 "question"이라고 가정할 때 값을 채워 넣습니다.
formatted_prompt = prompt.invoke({"question": "What is the color of the sky?"})

#

# 5. 실행
# ChatHuggingFace의 .invoke()는 LangChain의 PromptValue 객체를 직접 받을 수 있습니다.
# 따라서 OpenAI처럼 별도의 메시지 변환 함수가 필요하지 않아 코드가 더 깔끔해집니다.
response = model.invoke(formatted_prompt)

print("--- 모델 답변 ---")
print(response.content)

# Evaluation

In [ ]:
from langsmith import wrappers, Client
from pydantic import BaseModel, Field
from openai import OpenAI

client = Client()
openai_client = wrappers.wrap_openai(OpenAI())

# For other dataset creation methods, see: https://docs.smith.langchain.com/evaluation/how_to_guides/manage_datasets_programmatically https://docs.smith.langchain.com/evaluation/how_to_guides/manage_datasets_in_application

# Create inputs and reference outputs
examples = [
    (
        "Which country is Mount Kilimanjaro located in?",
        "Mount Kilimanjaro is located in Tanzania.",
    ),
    (
        "What is Earth's lowest point?",
        "Earth's lowest point is The Dead Sea.",
    ),
]

inputs = [{"question": input_prompt} for input_prompt, _ in examples]
outputs = [{"answer": output_answer} for _, output_answer in examples]

# Programmatically create a dataset in LangSmith
dataset = client.create_dataset(
    dataset_name = "Sample dataset",
    description = "A sample dataset in LangSmith."
)

# Add examples to the dataset
client.create_examples(inputs=inputs, outputs=outputs, dataset_id=dataset.id)

# Define the application logic you want to evaluate inside a target function
# The SDK will automatically send the inputs from the dataset to your target function
def target(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            { "role": "system", "content": "Answer the following question accurately" },
            { "role": "user", "content": inputs["question"] },
        ]
    )
    return { "response": response.choices[0].message.content.strip() } # type: ignore

# Define instructions for the LLM judge evaluator
instructions = """Evaluate Student Answer against Ground Truth for conceptual similarity and classify true or false:
- False: No conceptual match and similarity
- True: Most or full conceptual match and similarity
- Key criteria: Concept should match, not exact wording.
"""

# Define output schema for the LLM judge
class Grade(BaseModel):
    score: bool = Field(description="Boolean that indicates whether the response is accurate relative to the reference answer")

# Define LLM judge that grades the accuracy of the response relative to reference output
def accuracy(outputs: dict, reference_outputs: dict) -> bool:
    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            { "role": "system", "content": instructions },
            { "role": "user", "content": f"""Ground Truth answer: {reference_outputs["answer"]};
            Student's Answer: {outputs["response"]}"""
        }],
        response_format=Grade
    )
    return response.choices[0].message.parsed.score # type: ignore


# After running the evaluation, a link will be provided to view the results in langsmith
experiment_results = client.evaluate(
    target,
    data = "Sample dataset",
    evaluators = [
        accuracy, # type: ignore
        # can add multiple evaluators here
    ],
    experiment_prefix = "first-eval-in-langsmith",
    max_concurrency = 2,
) # type: ignore

print(experiment_results)

실행 후 LangSmith <Datasets & Experiments>에서 확인

In [ ]:
import os
from langsmith import Client
from pydantic import BaseModel, Field
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

client = Client()

# 2. 데이터셋 생성
examples = [
    ("Which country is Mount Kilimanjaro located in?", "Mount Kilimanjaro is located in Tanzania."),
    ("What is Earth's lowest point?", "Earth's lowest point is The Dead Sea."),
]

inputs = [{"question": input_prompt} for input_prompt, _ in examples]
outputs = [{"answer": output_answer} for _, output_answer in examples]

dataset_name = "HF Sample dataset v2"
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name=dataset_name, description="A sample dataset for HF.")
    client.create_examples(inputs=inputs, outputs=outputs, dataset_id=dataset.id)

# 3. 타겟 모델 설정 (평가 대상)
llm_ep = HuggingFaceEndpoint(repo_id="Qwen/Qwen2.5-7B-Instruct", task="text-generation")
model = ChatHuggingFace(llm=llm_ep)

def target(inputs: dict) -> dict:
    response = model.invoke([
        ("system", "Answer the following question accurately"),
        ("user", inputs["question"])
    ])
    return {"response": response.content.strip()}

# 4. LLM Judge(채점관) 및 구조화 설정
class Grade(BaseModel):
    score: bool = Field(description="Boolean score: True if accurate, False if not.")

# [핵심] Pydantic을 이용한 JSON 파서 설정
parser = JsonOutputParser(pydantic_object=Grade)

# 채점용 프롬프트 (JSON 형식을 강제하기 위한 format_instructions 포함)
judge_instructions = """Evaluate the Student Answer against the Ground Truth for conceptual similarity.
{format_instructions}

Ground Truth answer: {ground_truth}
Student's Answer: {student_answer}

Return ONLY the JSON object."""

judge_prompt = PromptTemplate(
    template=judge_instructions,
    input_variables=["ground_truth", "student_answer"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 채점용 모델 (타겟과 동일하거나 더 좋은 모델 권장)
judge_llm = ChatHuggingFace(llm=llm_ep)
# 체인 구성: 프롬프트 -> 모델 -> JSON 파서
judge_chain = judge_prompt | judge_llm | parser

def accuracy(outputs: dict, reference_outputs: dict) -> bool:
    """LLM Judge가 점수를 판정하는 함수"""
    try:
        result = judge_chain.invoke({
            "ground_truth": reference_outputs["answer"],
            "student_answer": outputs["response"]
        })
        # 결과에서 score 값을 추출 (dict 형식으로 반환됨)
        return result.get("score", False)
    except Exception as e:
        print(f"채점 파싱 에러 발생: {e}")
        return False

# 5. 평가 실행


print("--- 평가 시작 ---")
experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[accuracy],
    experiment_prefix="hf-json-eval",
    max_concurrency=2, # 무료 API 이용 시 동시성을 낮게 유지하는 것이 좋습니다.
)

print("\n--- 평가 완료 ---")
print(experiment_results)